# Ground Truth Metrics Evaluation

This notebook calculates key metrics (C_NPMI, C_V, D_PJD, D_PUW, and U_LLM) based on the ground truth labeled dataset.

These metrics provide a baseline understanding of the inherent topic quality in the gold labels.

## 1. Load Dependencies and Data

In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
from openai import OpenAI
from gensim.corpora import Dictionary
import gensim.corpora as corpora
from gensim.models.coherencemodel import CoherenceModel
from collections import Counter
from itertools import combinations
from pydantic import BaseModel
from tqdm import tqdm
import time

# Load environment variables
load_dotenv()
openai_api_key = os.getenv("OPENAI_KEY")
openai = OpenAI(api_key=openai_api_key)

# Define the metric functions directly here to avoid litellm import issues
# These are copied from textonomy.utils.metrics to avoid the circular import problem with litellm

def bt_topic_coherence_npmi(topic_words, texts, corpus, dictionary):
    """Compute the NPMI coherence of the topics"""
    cm_npmi = CoherenceModel(
        topics=topic_words, texts=tqdm(texts), corpus=corpus, dictionary=dictionary, coherence="c_npmi"
    )
    return cm_npmi.get_coherence()

def bt_topic_coherence_c_v(topic_words, texts, corpus, dictionary):
    """Compute the C_v coherence of the topics"""
    cm = CoherenceModel(
        topics=topic_words, texts=tqdm(texts), corpus=corpus, dictionary=dictionary, coherence="c_v"
    )
    return cm.get_coherence()

def bt_proportion_unique_words(topic_words, topk=10):
    """Compute the proportion of unique words"""
    if topk > len(topic_words[0]):
        raise Exception("Words in topics are less than " + str(topk))
    else:
        unique_words = set()
        for topic in topic_words:
            unique_words = unique_words.union(set(topic[:topk]))
        puw = len(unique_words) / (topk * len(topic_words))
        return puw

def bt_pairwise_jaccard_diversity(topic_words):
    """Compute the average pairwise jaccard distance between the topics"""
    dist = 0
    count = 0
    for list1, list2 in combinations(topic_words, 2):
        js = 1 - len(set(list1).intersection(set(list2))) / len(set(list1).union(set(list2)))
        dist = dist + js
        count = count + 1
    return dist / count

def bt_topic_usefulness(purpose_description: str, topic_model, openai_client):
    """Using an LLM to rate the usefulness of the topics"""
    
    class UsefulnessAnalysis(BaseModel):
        U_score: float
        U_reasoning: str

    retries = 5
    temperature = 1.0
    wait_time = 10

    rating_task_description = "Rate the usefulness of the following topic model and its produced topics for achieving a specific purpose. Consider the relevance, clarity, and comprehensiveness of the topics in relation to the purpose. Provide a score between 0 and 1, with 1 being the most useful. Explain your reasoning for the score. Be absolutely honest with your rating, delivering machine precision and not giving away good scores easily. Also take the distribution between topics and outliers into account. While a few outliers are normal and represent reality (~1-20%, depending on dataset and purpose, in some cases maybe even more) usually topic models with as few outliers as possible are best. The first topic in the table (-1) represents the outlier cluster/topic where all the not classified posts go. This is the defined purpose of the topic model:"

    message = [
        {"role": "user", "content": rating_task_description},
        {"role": "user", "content": purpose_description},
        {"role": "user", "content": "The produced topics of the topic model are as follows:"},
        {"role": "user", "content": str(topic_model.get_topic_info())},
    ]

    for attempt in range(retries):
        try:
            response = openai_client.beta.chat.completions.parse(
                model="gpt-4o-2024-08-06",
                messages=message,
                response_format=UsefulnessAnalysis,
                temperature=temperature,
            )
            print("Usefulness rating message:")
            print(message)
            print("Usefulness rating response:")
            print(response)
            parsed_response = response.choices[0].message.parsed
            return {
                field: getattr(parsed_response, field) for field in parsed_response.model_fields_set
            }
        except Exception as e:
            if attempt < retries - 1:
                print(f"Rate limit exceeded: {e}. Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print("Maximum retries reached. Raising the error.")
                raise
    return None

print("✓ Metric functions loaded successfully")

In [ ]:
# Configuration
dataset_name = "bills-hoyle-2022"  # or "wikitext-103-hoyle-2022"
gold_label_column = 'topic'  # or 'supercategory', 'category'

# Load data
test_data = pd.read_json(f"../data/{dataset_name}/test.metadata.jsonl", lines=True)
print(f"Loaded {len(test_data)} documents from {dataset_name}")
test_data.head()

In [ ]:
test_data.topic.nunique()

## 2. Preprocess Documents

In [ ]:
# Prepare tokenized documents for coherence metrics
input_column = "tokenized_text"

test_docs = test_data[input_column].tolist()
test_docs = [doc.split() for doc in test_docs]

print(f"Number of documents: {len(test_docs)}")
print(f"Sample tokenized doc (first 20 tokens): {test_docs[0][:20]}")

In [ ]:
# Create dictionary and corpus for coherence metrics
dictionary = Dictionary(test_docs)
corpus = [dictionary.doc2bow(doc) for doc in test_docs]

print(f'Number of unique tokens: {len(dictionary)}')
print(f'Number of documents in corpus: {len(corpus)}')

## 3. Extract Topic Words from Ground Truth Labels

For each ground truth topic/category, we'll extract the most representative words by computing TF-IDF or word frequency within that category.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

def extract_keywords_from_df(df, text_col="tokenized_text", topic_col="supercategory", top_n=10):
    """
    Extracts top N keywords for each topic using TF-IDF.

    Args:
        df (pd.DataFrame): DataFrame with a text column and a topic label column.
        text_col (str): Name of the column containing document text.
        topic_col (str): Name of the column with topic labels.
        top_n (int): Number of keywords per topic.

    Returns:
        dict: {topic: [keyword1, keyword2, ...]} with top keywords for each topic.
    """
    keywords_per_topic = {}

    for topic, group in df.groupby(topic_col):
        texts = group[text_col].astype(str).tolist()  # Ensure it's a list of strings
        vectorizer = TfidfVectorizer(stop_words='english', max_features=10_000)
        tfidf_matrix = vectorizer.fit_transform(texts)

        feature_names = vectorizer.get_feature_names_out()
        tfidf_scores = tfidf_matrix.mean(axis=0).A1  # Average TF-IDF per word

        top_keywords = [feature_names[i] for i in tfidf_scores.argsort()[-top_n:][::-1]]
        keywords_per_topic[topic] = top_keywords

    return keywords_per_topic

# Extract topic words from ground truth labels
topic_words = extract_keywords_from_df(test_data, text_col="tokenized_text", topic_col="topic", top_n=10)

print(f"Number of unique topics in ground truth: {len(topic_words)}")
print(f"\nResults:")
for i, (label, words) in enumerate(zip(topic_words.keys(), topic_words.values())):
    print(f"  {label}: {words}")

## 4. Calculate Metrics

Now we'll calculate the five key metrics:
- **C_NPMI**: Topic Coherence (Normalized Pointwise Mutual Information)
- **C_V**: Topic Coherence (C_v measure)
- **D_PJD**: Diversity - Pairwise Jaccard Distance
- **D_PUW**: Diversity - Proportion of Unique Words
- **U_LLM**: LLM-based Usefulness score

In [ ]:
# Calculate coherence metrics

topic_words_list = [i for i in topic_words.values()]
ref_corpus_texts = [doc.split() for doc in test_data['tokenized_text'].to_list()]


print("Calculating C_NPMI (Topic Coherence - NPMI)...")
c_npmi = bt_topic_coherence_npmi(topic_words_list, test_docs, corpus, dictionary)
print(f"C_NPMI: {c_npmi:.4f}")

print("\nCalculating C_V (Topic Coherence - C_v)...")
c_v = bt_topic_coherence_c_v(topic_words_list, test_docs, corpus, dictionary)
print(f"C_V: {c_v:.4f}")

In [ ]:
# Calculate diversity metrics
print("Calculating D_PUW (Proportion of Unique Words)...")
d_puw = bt_proportion_unique_words(topic_words_list, topk=10)
print(f"D_PUW: {d_puw:.4f}")

print("\nCalculating D_PJD (Pairwise Jaccard Distance)...")
d_pjd = bt_pairwise_jaccard_diversity(topic_words_list)
print(f"D_PJD: {d_pjd:.4f}")

In [ ]:
# Extract gold labels and unique labels for MockTopicModel
gold_labels = test_data[gold_label_column].tolist()
unique_labels = list(topic_words.keys())

print(f"Number of documents: {len(gold_labels)}")
print(f"Number of unique labels: {len(unique_labels)}")
print(f"Unique labels: {unique_labels}")

In [ ]:
# For U_LLM, we need to create a mock BERTopic-like object with topic info
# Since we're evaluating ground truth, we'll create a simple representation

class MockTopicModel:
    """Mock topic model for ground truth evaluation"""
    def __init__(self, topic_words, labels, label_names):
        self.topic_words = topic_words
        self.labels = labels
        self.label_names = label_names
    
    def get_topic_info(self):
        """Return topic info similar to BERTopic format"""
        topic_info = []
        label_counts = pd.Series(self.labels).value_counts()
        
        for i, label_name in enumerate(self.label_names):
            count = label_counts.get(label_name, 0)
            words = ', '.join(self.topic_words[i][:5])  # Top 5 words
            topic_info.append({
                'Topic': i,
                'Count': int(count),
                'Name': f"{i}_{label_name}",
                'Representation': words
            })
        
        return pd.DataFrame(topic_info)

# Create mock model
mock_model = MockTopicModel(topic_words_list, gold_labels, unique_labels)

# Display topic info
print("Ground Truth Topic Information:")
display(mock_model.get_topic_info())

In [ ]:
# Calculate LLM usefulness metric
purpose = "A taxonomy to organise these articles into main categories. Aim at around 10-20 categories. Examples: 'Music', 'Social sciences and society'..."

print("Calculating U_LLM (LLM Usefulness score)...")
print("Note: This may take a moment as it calls the OpenAI API...\n")

usefulness_result = bt_topic_usefulness(
    purpose_description=purpose,
    topic_model=mock_model,
    openai_client=openai
)

u_score = usefulness_result['U_score']
u_reasoning = usefulness_result['U_reasoning']

print(f"U_LLM Score: {u_score:.4f}")
print(f"\nReasoning:\n{u_reasoning}")

## 5. Summary of Results

In [ ]:
# Create summary dataframe
results = {
    'Metric': ['C_NPMI', 'C_V', 'D_PJD', 'D_PUW', 'U_LLM'],
    'Full Name': [
        'Topic Coherence (NPMI)',
        'Topic Coherence (C_v)',
        'Diversity - Pairwise Jaccard Distance',
        'Diversity - Proportion Unique Words',
        'LLM Usefulness Score'
    ],
    'Value': [c_npmi, c_v, d_pjd, d_puw, u_score],
    'Interpretation': [
        'Higher is better (coherence)',
        'Higher is better (coherence)',
        'Higher is better (more diverse topics)',
        'Higher is better (more diverse topics)',
        'Higher is better (more useful topics)'
    ]
}

results_df = pd.DataFrame(results)

print(f"\n{'='*80}")
print(f"GROUND TRUTH METRICS SUMMARY - {dataset_name}")
print(f"Gold Label Column: {gold_label_column}")
print(f"Number of Topics: {len(unique_labels)}")
print(f"Number of Documents: {len(test_data)}")
print(f"{'='*80}\n")

display(results_df)

# Save results
output_path = f"../data/tmp/ground_truth_metrics_{dataset_name}_{gold_label_column}.csv"
results_df.to_csv(output_path, index=False)
print(f"\nResults saved to: {output_path}")